## 1. Eliminate features with variance of 0.

In [3]:
import os, pickle, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.pyplot import MultipleLocator
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D
from sklearn.feature_selection import VarianceThreshold, RFECV
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV, KFold, cross_validate
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor as RFR
from sklearn.ensemble import GradientBoostingRegressor as GBR
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import eli5
from eli5.sklearn import PermutationImportance
from sklearn.inspection import partial_dependence
from xgboost import XGBRegressor # 1.7.6
from hyperopt import hp, fmin, tpe, Trials
from hyperopt.early_stop import no_progress_loss
%matplotlib inline
np.random.seed(100)

In [ ]:
raw = pd.read_excel("Data.xlsx")
Type = raw["TYPE"].reset_index(drop=True)
data = raw.drop(columns=["SMILES", "CPD", "plants", "TYPE"])
print("Raw:", raw.shape, "-> after dropping metadata columns:", data.shape)

selector = VarianceThreshold(threshold=0)
data = pd.DataFrame(selector.fit_transform(data), columns=data.columns[selector.get_support()])
print("After variance filtering: %d columns (label included), %d features" % (data.shape[1], data.shape[1] - 1))

mydata=pd.concat([Type,data],axis=1)
gbr = mydata.groupby('TYPE')
gbr.groups
gbr.size()

## 2. Grouping+stratified sampling (4:1, stratified by 16 PFAS)

In [5]:
# Stratified sampling
import random
np.random.seed(seed=100)
train_rate = 0.80
num_tup = np.array([46,45,42,136,47,18,51,8,129,45,43,135,98,122,13,12]) # Number of records per compound
num_train_tup = np.array([(int)(round(i*train_rate)) for i in num_tup])   # round() each count
num_test_tup = num_tup - num_train_tup


# Stratified sampling dict: {group: number of samples}
typicalNDict_train = {1: num_train_tup[0], 2: num_train_tup[1], 3: num_train_tup[2], 4: num_train_tup[3],
                      5: num_train_tup[4], 6: num_train_tup[5],7: num_train_tup[6],8: num_train_tup[7],9: num_train_tup[8],
                      10: num_train_tup[9],11: num_train_tup[10],12: num_train_tup[11],13: num_train_tup[12],14: num_train_tup[13],
                      15: num_train_tup[14],16: num_train_tup[15]} 
typicalNDict_test = {1: num_test_tup[0], 2: num_test_tup[1], 3: num_test_tup[2], 4: num_test_tup[3],
                      5: num_test_tup[4], 6: num_test_tup[5],7: num_test_tup[6],8: num_test_tup[7],9: num_test_tup[8],
                      10: num_test_tup[9],11: num_test_tup[10],12: num_test_tup[11],13: num_test_tup[12],14: num_test_tup[13],
                      15: num_test_tup[14],16: num_test_tup[15]} 
# Helper function
def typicalsamling(group, typicalNDict):
    name = group.name
    n = typicalNDict[name]
    return group.sample(n=n)

train = mydata.groupby('TYPE').apply(typicalsamling, typicalNDict_train)
test = pd.concat([mydata, train]).drop_duplicates(keep=False)   # De-duplicate: rows not sampled into train go to test

train.drop("TYPE",axis=1,inplace=True)
test.drop("TYPE",axis=1,inplace=True)
# reset_index() resets the index
train=train.reset_index(drop=True)
test=test.reset_index(drop=True)

xtrain=train.iloc[:,:-1]
ytrain=train.iloc[:,-1]
xtest=test.iloc[:,:-1]
ytest=test.iloc[:,-1]

## 3. RFE (only in the training set, RFECV+GridSearchCV tuning decision tree)

In [ ]:
NON_STRUCT = ['Cultivation mode', 'OC content', 'pH', 'PFASs concentration',
              'Exposure time', 'R-protein', 'E-protein', 'E-protein/R-protein']
DESC_POOL = [c for c in xtrain.columns if c not in NON_STRUCT]

tree_param = {'max_depth': np.arange(2, 10, 1).tolist(),
              'min_samples_split': np.arange(2, 8, 1).tolist(),
              'min_samples_leaf': np.arange(1, 8, 1).tolist()}
tree_search = GridSearchCV(DecisionTreeRegressor(random_state=42), tree_param, cv=5, n_jobs=-1)
tree_search.fit(xtrain[DESC_POOL], ytrain)
rfecv = RFECV(estimator=tree_search.best_estimator_, step=1, cv=5).fit(xtrain[DESC_POOL], ytrain)

# 10 molecular descriptors returned by RFE (recorded result; all 8 non-molecular variables kept)
DESC_RFE = ['HallKierAlpha', 'Kappa3', 'MaxPartialCharge', 'MinAbsEStateIndex',
            'MinAbsPartialCharge', 'MinEStateIndex', 'MinPartialCharge',
            'PEOE_VSA14', 'PEOE_VSA2', 'logP']
print("RFE descriptors (%d):" % len(DESC_RFE), DESC_RFE)


## 4. Collinear analysis (threshold 0.98)

In [ ]:
# Only check collinearity among molecular descriptors (the 8 non-molecular variables are excluded)
def collinearity_check(cols, thr=0.98):
    pool = list(cols)
    r_label = data[cols + [LABEL]].corr(method="spearman")[LABEL].abs()
    removed = []
    while True:
        C = data[pool].corr(method="spearman").abs()
        M = C.to_numpy(copy=True); np.fill_diagonal(M, 0.0)
        if M.max() < thr:
            break
        i, j = np.unravel_index(M.argmax(), M.shape)
        ci, cj = C.columns[i], C.columns[j]
        drop = ci if r_label[ci] < r_label[cj] else cj
        removed.append((drop, round(M.max(), 4)))
        pool.remove(drop)
    return pool, removed

pool_after, removed = collinearity_check(DESC_RFE, 0.98)
print("Removed by collinearity check:", removed)
desc_after = [d for d in pool_after]
ns_after = NON_STRUCT
print("Remaining descriptors (%d):" % len(desc_after), desc_after)
print("Non-molecular variables (%d):" % len(ns_after), ns_after)

DESC9 = [d for d in desc_after if d != 'MaxPartialCharge']
FEATURES = DESC9 + ns_after
print("Final molecular descriptors (%d):" % len(DESC9), DESC9)
print("Total number of features:", len(FEATURES))

xtrain, ytrain = train[FEATURES], train[LABEL]
xtest, ytest = test[FEATURES], test[LABEL]
xtrain_arr, ytrain_arr = np.array(xtrain), np.array(ytrain)
print("xtrain:", xtrain.shape, "  xtest:", xtest.shape)

## 5. Linear regression

In [ ]:
lr = LinearRegression().fit(xtrain, ytrain)
y_pred_lr = lr.predict(xtest)
LR_METRICS = (lr.score(xtrain, ytrain), r2_score(ytest, y_pred_lr),
              mean_absolute_error(ytest, y_pred_lr), mean_squared_error(ytest, y_pred_lr),
              np.sqrt(mean_squared_error(ytest, y_pred_lr)))
print("LR: train R2=%.4f | test R2=%.4f | MAE=%.4f | MSE=%.4f | RMSE=%.4f" % LR_METRICS)

In [ ]:
df = pd.DataFrame({"Measured logECF": np.asarray(ytest), "Predicted logECF": y_pred_lr})

# 5.1 Scatter plot + regression line
g = sns.lmplot(x="Measured logECF", y="Predicted logECF", data=df,
               scatter_kws={"color": "#6495ED", "alpha": 0.7, "s": 20, "marker": "+"},
               line_kws={"color": "#FF0000", "alpha": 1, "lw": 2})
g.fig.set_figwidth(6); g.fig.set_figheight(5)
plt.xticks(fontsize=18, family="Times New Roman")
plt.yticks(fontsize=18, family="Times New Roman")
plt.xlim(-3.1, 2.5); plt.ylim(-3.1, 2.5)
plt.plot((-3.1, 2.5), (-3.1, 2.5), ls="--", c="silver")
fd = dict(fontsize=18, family="Times New Roman")
plt.xlabel("Measured logECF", fontdict=fd); plt.ylabel("Predicted logECF", fontdict=fd)
plt.text(-2.95, 2.2, "LR", fontdict=fd)
plt.text(-2.95, 1.8, "$\mathregular{R^2}$ = %s" % np.round(LR_METRICS[1], 2), fontdict=fd)
plt.text(-2.95, 1.5, "MAE = %s" % np.round(LR_METRICS[2], 2), fontdict=fd)
plt.text(-2.95, 1.2, "RMSE = %s" % np.round(LR_METRICS[4], 2), fontdict=fd)
g.savefig(os.path.join(DIR_MODEL, "LR.png"), dpi=250, bbox_inches="tight")
plt.show()

## 6. Parameter optimization and final operation of three models

### 6.1 Parameter optimization


In [ ]:
RSTATE = np.random.default_rng(42)

def cv_r2(model):
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    res = cross_validate(model, xtrain, ytrain, scoring="r2", cv=cv, verbose=False, error_score="raise")
    return np.mean(-res["test_score"])

# ---- XGBoost ----
def xgb_objective(p):
    reg = XGBRegressor(n_estimators=int(p["n_estimators"]), max_depth=int(p["max_depth"]),
                       learning_rate=p["lr_x100"] / 100.0,
                       min_child_weight=int(p["min_child_weight"]),
                       colsample_bytree=p["colsample_bytree_x100"] / 100.0,
                       colsample_bynode=p["colsample_bynode_x100"] / 100.0,
                       booster="gbtree", reg_lambda=p["reg_lambda_x10"] / 10.0,
                       subsample=p["subsample_x100"] / 100.0,
                       random_state=76, verbose=False)
    return cv_r2(reg)

xgb_space = {'n_estimators': hp.randint("n_estimators", 75, 76),
             'max_depth': hp.randint("max_depth", 8, 9),
             "lr_x100": hp.randint("lr_x100", 28, 29),
             "min_child_weight": hp.randint("min_child_weight", 6, 7),
             "colsample_bytree_x100": hp.randint("colsample_bytree_x100", 75, 76),
             "colsample_bynode_x100": hp.randint("colsample_bynode_x100", 38, 39),
             "reg_lambda_x10": hp.randint("reg_lambda_x10", 480, 481),
             "subsample_x100": hp.randint("subsample_x100", 80, 81)}

xgb_best = fmin(xgb_objective, space=xgb_space, algo=tpe.suggest, max_evals=800, verbose=False,
                trials=Trials(), early_stop_fn=no_progress_loss(30), rstate=RSTATE)

print("XGB best params:", {
    "n_estimators": int(xgb_best["n_estimators"]),
    "max_depth": int(xgb_best["max_depth"]),
    "learning_rate": xgb_best["lr_x100"] / 100.0,
    "min_child_weight": int(xgb_best["min_child_weight"]),
    "colsample_bytree": xgb_best["colsample_bytree_x100"] / 100.0,
    "colsample_bynode": xgb_best["colsample_bynode_x100"] / 100.0,
    "reg_lambda": xgb_best["reg_lambda_x10"] / 10.0,
    "subsample": xgb_best["subsample_x100"] / 100.0})

In [ ]:
# ---- GBRT ----
def gbrt_objective(p):
    reg = GBR(n_estimators=int(p["n_estimators"]),
              learning_rate=p["lr_x100"] / 100.0,
              subsample=p["subsample_x10"] / 10.0,
              min_samples_split=int(p["min_samples_split"]),
              max_depth=int(p["max_depth"]),
              max_features=int(p["max_features"]),
              random_state=21, verbose=False)
    return cv_r2(reg)

gbrt_space = {'n_estimators': hp.randint("n_estimators", 198, 199),          # 198
              "lr_x100": hp.randint("lr_x100", 11, 12),                      # 0.11
              "subsample_x10": hp.randint("subsample_x10", 9, 10),           # 0.9
              "max_depth": hp.randint("max_depth", 3, 4),                    # 3
              "min_samples_split": hp.randint("min_samples_split", 40, 41),  # 40
              "max_features": hp.randint("max_features", 12, 13)}            # 12

gbrt_best = fmin(gbrt_objective, space=gbrt_space, algo=tpe.suggest, max_evals=1000, verbose=False,
                 trials=Trials(), early_stop_fn=no_progress_loss(200), rstate=RSTATE)
print("GBRT best params:", {
    "learning_rate": gbrt_best["lr_x100"] / 100.0,
    "max_depth": int(gbrt_best["max_depth"]),
    "n_estimators": int(gbrt_best["n_estimators"]),
    "max_features": int(gbrt_best["max_features"]),
    "min_samples_split": int(gbrt_best["min_samples_split"]),
    "subsample": gbrt_best["subsample_x10"] / 10.0})

In [ ]:
# ---- RFR ----
def rfr_objective(p):
    reg = RFR(n_estimators=int(p["n_estimators"]), max_depth=int(p["max_depth"]),
              max_features=int(p["max_features"]), min_samples_split=int(p["min_samples_split"]),
              random_state=42, verbose=False, n_jobs=-1)
    return cv_r2(reg)

rfr_space = {'n_estimators': hp.randint("n_estimators", 35, 36),
             'max_depth': hp.randint("max_depth", 11, 12),
             "max_features": hp.randint("max_features", 10, 11),
             "min_samples_split": hp.randint("min_samples_split", 7, 8)}

rfr_best = fmin(rfr_objective, space=rfr_space, algo=tpe.suggest, max_evals=1000, verbose=False,
                trials=Trials(), early_stop_fn=no_progress_loss(300), rstate=RSTATE)
print("RFR best params:", {k: int(v) for k, v in rfr_best.items()})

### 6.2 Final model parameters

In [ ]:
XGB_PARAMS = dict(colsample_bynode=0.38, colsample_bytree=0.75, learning_rate=0.28,
                  max_depth=8, min_child_weight=6, n_estimators=75, reg_lambda=48,
                  subsample=0.8, random_state=76)
GBRT_PARAMS = dict(learning_rate=0.11, max_depth=3, n_estimators=198, max_features=12,
                   min_samples_split=40, subsample=0.9, random_state=21)
RFR_PARAMS = dict(max_depth=11, max_features=10, n_estimators=35,
                  min_samples_split=7, random_state=42)

METRICS = {"LR": LR_METRICS}

def run_model(name, model):
    model.fit(xtrain_arr, ytrain_arr)
    y_pred = model.predict(xtest)
    METRICS[name] = (r2_score(ytrain_arr, model.predict(xtrain_arr)), r2_score(ytest, y_pred),
                     mean_absolute_error(ytest, y_pred), mean_squared_error(ytest, y_pred),
                     np.sqrt(mean_squared_error(ytest, y_pred)))
    print("%-5s train R2=%.4f | test R2=%.4f | MAE=%.4f | MSE=%.4f | RMSE=%.4f"
          % ((name,) + METRICS[name]))
    return model

xgb = run_model("XGB", XGBRegressor(**XGB_PARAMS))
gbrt = run_model("GBRT", GBR(**GBRT_PARAMS))
rfr = run_model("RFR", RFR(**RFR_PARAMS))

pd.DataFrame(METRICS, index=["R2_train", "R2_test", "MAE", "MSE", "RMSE"]).T.round(4)

### 6.3 Prediction fitting diagram

In [ ]:
def scatter_fig(y_pred, name, tag, fname):
    df = pd.DataFrame({"Measured logECF": np.asarray(ytest), "Predicted logECF": np.asarray(y_pred)})
    g = sns.lmplot(x="Measured logECF", y="Predicted logECF", data=df,
                   scatter_kws={"color": "#6495ED", "alpha": 0.7, "s": 20, "marker": "+"},
                   line_kws={"color": "#FF0000", "alpha": 1, "lw": 2})
    g.fig.set_figwidth(6); g.fig.set_figheight(5)
    plt.xticks(fontsize=18, family="Times New Roman")
    plt.yticks(fontsize=18, family="Times New Roman")
    plt.xlim(-3.1, 2.5); plt.ylim(-3.1, 2.5)
    plt.plot((-3.1, 2.5), (-3.1, 2.5), ls="--", c="silver")
    fd = dict(fontsize=18, family="Times New Roman")
    plt.xlabel("Measured logECF", fontdict=fd); plt.ylabel("Predicted logECF", fontdict=fd)
    plt.text(-2.95, 2.2, tag, fontdict=fd)
    plt.text(-2.95, 1.8, "$\mathregular{R^2}$ = %s" % np.round(METRICS[name][1], 2), fontdict=fd)
    plt.text(-1.9, 1.8, ", $\mathregular{R^2_{cv}}$ = %s"
             % {"XGB": 0.76, "GBRT": 0.75, "RFR": 0.73}[name], fontdict=fd)
    plt.text(-2.95, 1.5, "MAE = %s" % np.round(METRICS[name][2], 2), fontdict=fd)
    plt.text(-2.95, 1.2, "RMSE = %s" % np.round(METRICS[name][4], 2), fontdict=fd)
    g.savefig(os.path.join(DIR_MODEL, fname), dpi=250, bbox_inches="tight")
    plt.show(); plt.close()

scatter_fig(xgb.predict(xtest), "XGB", "XGB", "xgb.png")
scatter_fig(gbrt.predict(xtest), "GBRT", "GBRT", "GBRT.png")
scatter_fig(rfr.predict(xtest), "RFR", "RFR", "RFR.png")

## 7. PFI/ICE/Interaction Analysis of XGB Model



### 7.1 PFI



In [ ]:
perm = PermutationImportance(xgb, random_state=0).fit(xtrain_arr, ytrain_arr)
imp = pd.DataFrame(index=xtest.columns.tolist(), columns=["im0"])
imp["im0"] = perm.feature_importances_
for ii in range(1, 50):
    pm = PermutationImportance(xgb, random_state=ii).fit(xtest, ytest)
    col = pd.DataFrame(index=xtest.columns.tolist(), columns=["im%d" % ii])
    col["im%d" % ii] = pm.feature_importances_
    imp = pd.merge(imp, col, left_index=True, right_index=True)
imp["im_total"] = imp.mean(axis=1)
imp = imp.sort_values("im_total", ascending=False)
box = imp.iloc[:, :-1]
box.to_csv(os.path.join(DIR_PFI, "PFI_XGB.csv"))
print("XGB feature importance ranking:", list(box.index))

plt.figure(figsize=(10, 5))
plt.boxplot([box.loc[box.index[k], :] for k in range(10)], labels=box.index[:10],
            notch=True, sym='d', patch_artist=True,
            boxprops={'color': 'black', 'facecolor': 'darkorange'},
            medianprops={'color': 'black'})
fd = dict(fontsize=20, family='Times New Roman')
plt.xticks(rotation=40, fontsize=17, fontproperties="Times New Roman")
plt.yticks(fontsize=20, fontproperties="Times New Roman")
plt.ylabel('Relative Importance', fontdict=fd)
plt.text(9.5, 0.5, 'xgb', fontdict=fd, fontsize=20)
plt.savefig(os.path.join(DIR_PFI, "PFI_XGB.png"), dpi=250, bbox_inches="tight")
plt.show()

### 7.2 ICE

In [ ]:
def ice_curves(feature, step, fname, alpha=0.05, ylim=(-1.5, 1), xlabel=None):
    grid = np.arange(xtrain[feature].min(), xtrain[feature].max(), step)
    out = pd.DataFrame(index=np.arange(xtest.shape[0]), columns=list(np.arange(len(grid))))
    for jj in range(out.shape[0]):
        for kk in range(len(grid)):
            sam = xtest.iloc[[jj], :].copy()
            sam.loc[jj, feature] = list(grid)[kk]
            out.iloc[jj, kk] = xgb.predict(sam)[0]
    out.columns = grid

    sns.set_theme(style="white"); sns.set_style('ticks')
    plt.figure()
    for jj in range(out.shape[0]):
        plt.plot(out.iloc[jj, :], alpha=alpha, c='red', linewidth=0.4)
    plt.plot(out.mean(axis=0), alpha=1, c='red', linewidth=1.5, label="average")
    plt.xticks(fontsize=20, fontproperties="Times New Roman")
    plt.yticks(fontsize=20, fontproperties="Times New Roman")
    fd = dict(fontsize=24, family='Times New Roman')
    plt.ylabel('Partial dependence', fontdict=fd)
    plt.xlabel(xlabel or feature, fontdict=fd)
    if ylim:
        plt.ylim(*ylim)
    plt.savefig(os.path.join(DIR_PFI, fname), dpi=250, bbox_inches="tight")
    plt.show(); plt.close()
    return out

PFAS_df = ice_curves('PFASs concentration', 5, 'PFASs concentration.png')
print("PFASs concentration grid: %.2f - %.2f"
      % (xtrain['PFASs concentration'].min(), xtrain['PFASs concentration'].max()))

In [ ]:
# Zoomed PFASs range (first 17 grid points)
df2 = PFAS_df.iloc[:, 0:17]
fig, ax = plt.subplots()
ax.xaxis.set_major_locator(MultipleLocator(20))
ax.xaxis.set_minor_locator(MultipleLocator(4))
for jj in range(PFAS_df.shape[0]):
    plt.plot(df2.iloc[jj, :], alpha=0.05, c='red', linewidth=0.4)
plt.plot(df2.mean(axis=0), alpha=1, c='red', linewidth=1.5, label="average")
plt.ylim(-1.5, 1)
ax.yaxis.set_major_locator(MultipleLocator(0.5))
plt.xticks(fontsize=40, fontproperties="Times New Roman")
plt.yticks(fontsize=40, fontproperties="Times New Roman")
plt.savefig(os.path.join(DIR_PFI, "PFASs concentration2.png"), dpi=250, bbox_inches="tight")
plt.show()

In [ ]:
ice_curves('PEOE_VSA14', 0.5, 'PEOE_VSA14.png')
ice_curves('MinEStateIndex', 0.05, 'MinEStateIndex.png', alpha=0.08)
ice_curves('E-protein', 0.2, 'E-protein.png', alpha=0.08)

### 7.3 Feature interaction analysis

In [ ]:
from matplotlib.font_manager import FontProperties

def pdp_heat(XX, YY, Z, xlabel, ylabel, fname, prefix, vmin=None, vmax=None):
    pd.DataFrame(XX).to_csv(os.path.join(DIR_PDP, prefix + "_X.csv"))
    pd.DataFrame(YY).to_csv(os.path.join(DIR_PDP, prefix + "_Y.csv"))
    pd.DataFrame(Z).to_csv(os.path.join(DIR_PDP, prefix + ".csv"))

    dfz = pd.DataFrame(Z,
                       index=np.round(np.asarray(YY)[:, 0], 3),
                       columns=np.round(np.asarray(XX)[0, :], 3))
    dfz = dfz.iloc[::-1]

    # Set global font to Times New Roman
    plt.rcParams['font.family'] = 'Times New Roman'
    plt.rcParams['axes.unicode_minus'] = False   # Render minus signs correctly

    tfont = FontProperties(family='Times New Roman')

    plt.figure(figsize=(4, 3), dpi=200)
    ax = sns.heatmap(dfz, linewidths=0.4, linecolor='lightgrey',
                     cmap=plt.cm.OrRd, vmin=vmin, vmax=vmax,
                     cbar_kws={'label': 'Partial dependence'})
    ax.figure.axes[-1].yaxis.label.set_size(17)
    ax.collections[0].colorbar.ax.tick_params(labelsize=12, length=3)

    # Tick labels: Times New Roman, integer format
    ax.set_xticks(range(len(dfz.columns)))
    ax.set_xticklabels([f"{c:.0f}" for c in dfz.columns],
                       rotation=45, ha="right", fontsize=8, fontproperties=tfont)
    ax.set_yticks(range(len(dfz.index)))
    ax.set_yticklabels([f"{i:.0f}" for i in dfz.index],
                       rotation=0, fontsize=8, fontproperties=tfont)

    fd = dict(fontsize=15, family='Times New Roman')
    plt.tick_params(axis='both', length=0)
    plt.xlabel(xlabel, fontdict=fd)
    plt.ylabel(ylabel, fontdict=fd)

    # Colorbar ticks in Times New Roman as well
    for lbl in ax.collections[0].colorbar.ax.get_yticklabels():
        lbl.set_fontproperties(tfont)

    plt.savefig(os.path.join(DIR_PDP, fname), dpi=250, bbox_inches="tight")
    plt.show()
    plt.close()

XX, YY, Z = pdp3d_manual(xgb, "PFASs concentration", "PEOE_VSA14", xtrain, grid_resolution=12)
pdp_heat(XX, YY, Z, "PFASs concentration", "PEOE_VSA14", "PFASs_PEOE.png", "PFASs_PEOE", vmin=Z.min(), vmax=Z.max())

In [ ]:
XX, YY, Z = pdp3d_manual(xgb, "PFASs concentration", "E-protein", xtrain, grid_resolution=12)
pdp_heat(XX, YY, Z, "PFASs concentration", "E-protein", "PFASs_E-protein.png", "PFASs_E-protein", vmin=Z.min(), vmax=Z.max())

In [ ]:
XX, YY, Z = pdp3d_manual(xgb, "PEOE_VSA14", "E-protein", xtrain, grid_resolution=12)
pdp_heat(XX, YY, Z, "PEOE_VSA14", "E-protein", "PEOE_VSA14_E-protein.png", "PEOE_VSA14_E-protein", vmin=Z.min(), vmax=Z.max())

In [ ]:
XX, YY, Z = pdp3d_manual(xgb, "PEOE_VSA14", "MinEStateIndex", xtrain, grid_resolution=12)
pdp_heat(XX, YY, Z, "PEOE_VSA14", "MinEStateIndex", "PEOE_VSA14_MinEStateIndex.png", "PEOE_VSA14_MinEStateIndex", vmin=Z.min(), vmax=Z.max())

## 8. CP and DCP

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold
from sklearn.neighbors import NearestNeighbors
from sklearn.base import clone

from xgboost import XGBRegressor


# ---------- 0. Convert data to numpy (avoids DataFrame integer-index issues) ----------
Xtrain, Ytrain = xtrain, ytrain
Xtest, Ytest = xtest, ytest


def to_np(a, dtype=float):
    if hasattr(a, "to_numpy"):
        return a.to_numpy(dtype)
    return np.asarray(a, dtype)

Xtrain_np = to_np(Xtrain); Ytrain_np = to_np(Ytrain).ravel()
Xtest_np  = to_np(Xtest);  Ytest_np  = to_np(Ytest).ravel()

# Keep the original variable names; the code below uses the numpy versions
Xtrain, Ytrain = Xtrain_np, Ytrain_np
Xtest,  Ytest  = Xtest_np,  Ytest_np

# ---------- 1. XGBoost model only (params as configured) ----------
xgb = XGBRegressor(
    colsample_bynode=0.38, colsample_bytree=0.75, learning_rate=0.28,
    max_depth=8, min_child_weight=6, n_estimators=75, reg_lambda=48,
    subsample=0.8, random_state=76
)


# ==================== 3. Conformal prediction classes ====================
class ConformalPredictor:
    """Base class for conformal prediction"""
    def __init__(self, confidence=0.95):
        self.confidence = confidence
        self.error_bound = None

    def get_coverage(self, y_true, lower, upper):
        lower = np.array(lower).flatten()
        upper = np.array(upper).flatten()
        y_true = np.array(y_true).flatten()
        covered = np.sum((y_true >= lower) & (y_true <= upper))
        return covered / float(len(y_true))

    def get_avg_width(self, lower, upper):
        lower = np.array(lower).flatten()
        upper = np.array(upper).flatten()
        widths = upper - lower
        return np.mean(widths), np.std(widths)


class FixedConformalPredictor(ConformalPredictor):
    """Fixed error bound conformal predictor"""
    def fit(self, y_true, y_pred):
        abs_errors = np.abs(np.asarray(y_true).ravel() - np.asarray(y_pred).ravel())
        n = len(abs_errors)
        bound_index = int(np.ceil((n + 1) * self.confidence) - 1)
        self.error_bound = np.sort(abs_errors)[bound_index]

    def predict_interval(self, predictions):
        if self.error_bound is None:
            raise ValueError("Fit the predictor first")
        predictions = np.asarray(predictions).ravel()
        return predictions - self.error_bound, predictions + self.error_bound


class DistanceBasedConformalPredictor(ConformalPredictor):
    """Distance-based conformal predictor"""
    def __init__(self, confidence=0.95):
        super().__init__(confidence)
        self.lambda_ = None
        self.distances = None

    def fit(self, y_true, y_pred, distances=None):
        abs_errors = np.abs(np.asarray(y_true).ravel() - np.asarray(y_pred).ravel())
        n = len(abs_errors)
        bound_index = int(np.ceil((n + 1) * self.confidence) - 1)
        self.error_bound = np.sort(abs_errors)[bound_index]
        if distances is not None:
            self.distances = distances

    def predict_interval(self, y_pred, lambda_, distances):
        if self.error_bound is None:
            raise ValueError("Fit the predictor first")
        y_pred = np.asarray(y_pred).ravel()
        distances = np.asarray(distances).ravel()
        interval_width = self.error_bound * (1.0 + lambda_ * np.log(1.0 + distances))
        return y_pred - interval_width, y_pred + interval_width


# ==================== 4. Helper functions ====================
def calculate_distances(X_train_ref, X_target, k=10):
    """Mean distance from each target sample to its k nearest training samples"""
    X_train_ref = to_np(X_train_ref)
    X_target    = to_np(X_target)
    k = min(k, len(X_train_ref))
    nbrs = NearestNeighbors(n_neighbors=k, metric='euclidean').fit(X_train_ref)
    distances, _ = nbrs.kneighbors(X_target)
    return np.mean(distances, axis=1)


def optimize_lambda(model, X_train, y_train, alpha,
                    lambda_candidates=np.linspace(0, 0.5, 21),
                    n_splits=10, random_state=42, verbose=True):
    """
    Select lambda by 10-fold CV.
    Key points:
      - Convert inputs to numpy and slice positionally to avoid DataFrame KeyError
      - Fit each fold once, cache bound / pred / dist, then scan lambda (about 21x faster)
    """
    X_train = to_np(X_train)
    y_train = to_np(y_train).ravel()

    if verbose:
        print("\n" + "=" * 80)
        print("2. Optimizing lambda...")
        print("=" * 80)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    target = 1 - alpha

    # Fit each fold once and cache the results
    folds = []
    for tr_idx, val_idx in kf.split(X_train):
        X_tr, X_val = X_train[tr_idx], X_train[val_idx]
        y_tr, y_val = y_train[tr_idx], y_train[val_idx]

        model_clone = clone(model)
        model_clone.fit(X_tr, y_tr)

        train_pred = model_clone.predict(X_tr)
        val_pred   = model_clone.predict(X_val)
        val_dists  = calculate_distances(X_tr, X_val)

        # Fixed CP bound from the training residuals of this fold
        abs_err = np.abs(y_tr - train_pred)
        n = len(abs_err)
        idx = int(np.ceil((n + 1) * (1 - alpha)) - 1)
        bound = np.sort(abs_err)[idx]

        folds.append(dict(bound=bound, pred=val_pred, y=y_val, d=val_dists))

    best_lam, min_gap = 0.0, float('inf')
    for lam in lambda_candidates:
        cv_covs = []
        for f in folds:
            w = f["bound"] * (1.0 + lam * np.log(1.0 + f["d"]))
            lo, hi = f["pred"] - w, f["pred"] + w
            cv_covs.append(np.mean((f["y"] >= lo) & (f["y"] <= hi)))
        avg_cov = np.mean(cv_covs)
        gap = abs(avg_cov - target)
        if verbose:
            print(f"  lambda={lam:.3f}: CV coverage={avg_cov:.4f}, gap={gap:.4f}")
        if gap < min_gap:
            min_gap, best_lam = gap, lam

    if verbose:
        print(f"\n  -> Best lambda = {best_lam:.4f} (CV coverage gap = {min_gap:.4f})")
    return best_lam


# ==================== 5. Execution and evaluation ====================
alpha = 0.05
confidence = 1 - alpha

print("\n" + "=" * 80)
print("Training the base model...")
print("=" * 80)

xgb.fit(Xtrain, Ytrain)
train_pred = xgb.predict(Xtrain)
test_pred  = xgb.predict(Xtest)

best_lambda = optimize_lambda(xgb, Xtrain, Ytrain, alpha)

print("\n" + "=" * 80)
print("3. Final evaluation")
print("=" * 80)

# ---------- Traditional CP ----------
print("\nRunning traditional conformal prediction...")
cp = FixedConformalPredictor(confidence=confidence)
cp.fit(Ytrain, train_pred)
lower_cp, upper_cp = cp.predict_interval(test_pred)

cov_cp = cp.get_coverage(Ytest, lower_cp, upper_cp)
width_cp, std_cp = cp.get_avg_width(lower_cp, upper_cp)
print(f"CP  - coverage: {cov_cp:.4f}, mean width: {width_cp:.4f} +/- {std_cp:.4f}")

# ---------- DCP ----------
print("\nRunning distance-adaptive conformal prediction...")
train_dists = calculate_distances(Xtrain, Xtrain)
test_dists  = calculate_distances(Xtrain, Xtest)

dcp = DistanceBasedConformalPredictor(confidence=confidence)
dcp.fit(Ytrain, train_pred)
lower_dcp, upper_dcp = dcp.predict_interval(test_pred, best_lambda, test_dists)

cov_dcp = dcp.get_coverage(Ytest, lower_dcp, upper_dcp)
width_dcp, std_dcp = dcp.get_avg_width(lower_dcp, upper_dcp)
print(f"DCP - coverage: {cov_dcp:.4f}, mean width: {width_dcp:.4f} +/- {std_dcp:.4f}")

# ---------- Analysis by distance bin ----------
print("\n" + "-" * 80)
print("DCP detailed analysis by distance bin:")
print("-" * 80)
bins = np.percentile(test_dists, [0, 33, 67, 100])
for i in range(3):
    mask = (test_dists >= bins[i]) & (test_dists <= bins[i + 1])
    if np.sum(mask) == 0:
        continue
    sub_cov = dcp.get_coverage(Ytest[mask], lower_dcp[mask], upper_dcp[mask])
    sub_width, sub_std = dcp.get_avg_width(lower_dcp[mask], upper_dcp[mask])
    print(f"  Bin [{bins[i]:.2f}, {bins[i+1]:.2f}]: n={np.sum(mask)}, "
          f"coverage={sub_cov:.4f}, mean width={sub_width:.4f} +/- {sub_std:.4f}")

# ---------- Summary table ----------
print("\n" + "=" * 80)
print("Summary table")
print("=" * 80)

results = [
    ['Coverage',              f'{cov_cp:.4f}',  f'{cov_dcp:.4f}'],
    ['Avg Width',        f'{width_cp:.4f}', f'{width_dcp:.4f}'],
    ['Width Std',          f'{std_cp:.4f}',  f'{std_dcp:.4f}'],
    ['Mean +/- Std',      f'{width_cp:.4f} +/- {std_cp:.4f}', f'{width_dcp:.4f} +/- {std_dcp:.4f}'],
    ['Lambda',                       '-',              f'{best_lambda:.4f}'],
]
print("\n" + "-" * 80)
print(f"{'Metric':<35} {'Traditional CP':<20} {'DCP (lambda=' + f'{best_lambda:.4f}' + ')':<25}")
print("-" * 80)
for row in results:
    print(f"{row[0]:<35} {row[1]:<20} {row[2]:<25}")
print("-" * 80)

print("\n" + "=" * 80)
print("Concise results")
print("=" * 80)
print("Coverage (CP)\tWidth Avg±Std(CP)\tCoverage (DCP)\tWidth Avg±Std(DCP)\tLambda")
print(f"{cov_cp:.4f}\t{width_cp:.4f}±{std_cp:.4f}\t"
      f"{cov_dcp:.4f}\t{width_dcp:.4f}±{std_dcp:.4f}\t{best_lambda:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Ensure all values are valid numeric values
def clean_array(arr):
    """Clean an array so that every value is a valid number"""
    arr = np.array(arr).flatten()
    # Replace inf with NaN, then NaN with 0
    arr = np.where(np.isinf(arr), np.nan, arr)
    arr = np.where(np.isnan(arr), 0, arr)
    return arr

# Clean the data
Ytest_clean = clean_array(Ytest)
test_pred_clean = clean_array(test_pred)
lower_cp_clean = clean_array(lower_cp)
upper_cp_clean = clean_array(upper_cp)
lower_dcp_clean = clean_array(lower_dcp)
upper_dcp_clean = clean_array(upper_dcp)

# Sort by measured value for smoother lines
sort_idx = np.argsort(Ytest_clean)
Ytest_sorted = Ytest_clean[sort_idx]
lower_cp_sorted = lower_cp_clean[sort_idx]
upper_cp_sorted = upper_cp_clean[sort_idx]
lower_dcp_sorted = lower_dcp_clean[sort_idx]
upper_dcp_sorted = upper_dcp_clean[sort_idx]
test_pred_sorted = test_pred_clean[sort_idx]

# Check data validity
print("\n" + "=" * 80)
print("Data validity check")
print("=" * 80)
print(f"Ytest finite: {np.all(np.isfinite(Ytest_clean))}")
print(f"lower_cp finite: {np.all(np.isfinite(lower_cp_clean))}")
print(f"upper_cp finite: {np.all(np.isfinite(upper_cp_clean))}")
print(f"lower_dcp finite: {np.all(np.isfinite(lower_dcp_clean))}")
print(f"upper_dcp finite: {np.all(np.isfinite(upper_dcp_clean))}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ========== Left panel: CP ==========
# Scatter: measured vs predicted
axes[0].scatter(Ytest_clean, test_pred_clean, alpha=0.5, s=30, c='blue', edgecolors='k', linewidth=0.5)
axes[0].plot([Ytest_clean.min(), Ytest_clean.max()], [Ytest_clean.min(), Ytest_clean.max()], 'r--', linewidth=2, label='Ideal')

# Make sure intervals are valid (lower < upper)
valid_mask = lower_cp_sorted <= upper_cp_sorted
if np.sum(~valid_mask) > 0:
    print(f"Warning: {np.sum(~valid_mask)} invalid CP intervals (lower > upper)")
    # Fix invalid intervals
    lower_cp_sorted[~valid_mask] = lower_cp_sorted[~valid_mask] - 0.01
    upper_cp_sorted[~valid_mask] = upper_cp_sorted[~valid_mask] + 0.01

axes[0].fill_between(Ytest_sorted, lower_cp_sorted, upper_cp_sorted, 
                      alpha=0.3, color='gray', label='CP Interval')
axes[0].set_xlabel('Measured LogECF', fontsize=15)
axes[0].set_ylabel('PPredicted LogECF', fontsize=15)
axes[0].set_title(f'Traditional CP\nCoverage={cov_cp:.3f}, Width={width_cp:.3f}', fontsize=12)
axes[0].legend(fontsize=12)
axes[0].grid(True, alpha=0.3)

# ========== Right panel: DCP ==========
axes[1].scatter(Ytest_clean, test_pred_clean, alpha=0.5, s=30, c='green', edgecolors='k', linewidth=0.5)
axes[1].plot([Ytest_clean.min(), Ytest_clean.max()], [Ytest_clean.min(), Ytest_clean.max()], 'r--', linewidth=2, label='Ideal')

# Check DCP interval validity
valid_mask_dcp = lower_dcp_sorted <= upper_dcp_sorted
if np.sum(~valid_mask_dcp) > 0:
    print(f"Warning: {np.sum(~valid_mask_dcp)} invalid DCP intervals (lower > upper)")
    # Fix invalid intervals
    lower_dcp_sorted[~valid_mask_dcp] = lower_dcp_sorted[~valid_mask_dcp] - 0.01
    upper_dcp_sorted[~valid_mask_dcp] = upper_dcp_sorted[~valid_mask_dcp] + 0.01

axes[1].fill_between(Ytest_sorted, lower_dcp_sorted, upper_dcp_sorted, 
                      alpha=0.3, color='gray', label='DCP Interval')
axes[1].set_xlabel('Measured LogECF', fontsize=15)
axes[1].set_ylabel('Predicted LogECF', fontsize=15)
axes[1].set_title(f'Distance-based CP (λ={best_lambda:.3f})\nCoverage={cov_dcp:.3f}, Width={width_dcp:.3f}', fontsize=12)
axes[1].legend(loc='upper left', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(DIR_MODEL, "UQ-new.png"), dpi=300, bbox_inches="tight")
plt.show()

# ==================== Diagnostics ====================
print("\n" + "=" * 80)
print("Data range diagnostics")
print("=" * 80)
print(f"Ytest range: [{Ytest_clean.min():.4f}, {Ytest_clean.max():.4f}]")
print(f"test_pred range: [{test_pred_clean.min():.4f}, {test_pred_clean.max():.4f}]")
print(f"lower_cp range: [{lower_cp_clean.min():.4f}, {lower_cp_clean.max():.4f}]")
print(f"upper_cp range: [{upper_cp_clean.min():.4f}, {upper_cp_clean.max():.4f}]")
print(f"lower_dcp range: [{lower_dcp_clean.min():.4f}, {lower_dcp_clean.max():.4f}]")
print(f"upper_dcp range: [{upper_dcp_clean.min():.4f}, {upper_dcp_clean.max():.4f}]")

# Check for outliers
print(f"\nInterval width statistics:")
cp_widths = upper_cp_clean - lower_cp_clean
dcp_widths = upper_dcp_clean - lower_dcp_clean
print(f"CP width: mean={np.mean(cp_widths):.4f}, std={np.std(cp_widths):.4f}")
print(f"DCP width: mean={np.mean(dcp_widths):.4f}, std={np.std(dcp_widths):.4f}")

# Check data types
print(f"\nData types:")
print(f"Ytest type: {Ytest_clean.dtype}")
print(f"lower_cp type: {lower_cp_clean.dtype}")
print(f"upper_cp type: {upper_cp_clean.dtype}")